In [0]:
import io
import json
import logging
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from PIL import Image
from confluent_kafka.schema_registry import SchemaRegistryClient
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.avro.functions import from_avro
from pyspark.sql.streaming import DataStreamWriter, StreamingQuery
from pyspark.sql.types import StructType

# Configuración de logging para ejecución en cluster o job
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class LandingBatchReader:
    """
    Clase para leer archivos microbatch o batch desde una ruta landing, usando Auto Loader de Databricks.
    Permite añadir columnas de metadatos y controlar los formatos de archivo permitidos.
    """

    def __init__(
        self,
        data_source: str,
        dataset: str,
        landing_path: str,
        raw_path: str,
        bronze_path: str,
        dataset_format: str,
        schema_json: Optional[Dict[str, Any]],
        allowed_file_formats: List[str],
        processing_type: str
    ):
        """
        Inicializa el lector de la zona landing.
        Args:
            data_source: Nombre de la fuente de datos.
            dataset: Nombre del dataset.
            landing_path: Ruta del almacenamiento landing.
            raw_path: Ruta de almacenamiento raw.
            bronze_path: Ruta de almacenamiento bronze.
            dataset_format: Formato de los archivos (json, csv, parquet, etc.).
            schema_json: Esquema del dataset en formato dict.
            allowed_file_formats: Lista de extensiones de archivos permitidos.
        """
        self.data_source = data_source
        self.dataset = dataset
        self.format = dataset_format
        # si el formato es binary, no tiene esquema
        if schema_json:
            self.schema_json = StructType.fromJson(schema_json)
        self.landing_path = landing_path
        self.raw_path = raw_path
        self.bronze_path = bronze_path
        self.bronze_schema_location = f'{self.bronze_path}_schema'
        self.processing_type = processing_type

    def __str__(self) -> str:
        """Devuelve una representación legible de la instancia."""
        return (
            f"[LandingBatchReader] dataset='{self.dataset}' "
            f"source='{self.data_source}' format='{self.format}' "
            f"paths: landing='{self.landing_path}', raw='{self.raw_path}', bronze='{self.bronze_path}' "
            f"schema: {self.schema_json if self.schema_json else 'no'}"
        )

    @staticmethod
    def validate_config(
            config: Dict[str, Any],
            data_source: str,
            dataset: str
    ) -> None:
        """
        Valida la existencia de los parámetros obligatorios para un dataset en la configuración.
        Args:
            config: Configuración completa cargada (dict).
            data_source: Nombre de la fuente de datos.
            dataset: Nombre del dataset.
        Raises:
            KeyError: Si falta cualquier parámetro obligatorio.
        """
        try:
            if 'data_type_information' not in config:
                raise KeyError("Falta la clave 'data_type_information' en el config.")
            if data_source not in config['data_type_information']:
                raise KeyError(f"Falta el data_source '{data_source}' en 'data_type_information'.")
            if dataset not in config['data_type_information'][data_source]:
                raise KeyError(f"Falta el dataset '{dataset}' en data_source '{data_source}'.")

            dataset_conf = config['data_type_information'][data_source][dataset]
            required_fields = [
                'landing_path',
                'raw_path',
                'bronze_path',
                'dataset_format',
                'partition_columns'
            ]
            for field in required_fields:
                if field not in dataset_conf:
                    raise KeyError(f"Falta el campo '{field}' en el dataset '{dataset}' de la fuente '{data_source}'.")

            if 'allowed_file_formats' not in config:
                raise KeyError("Falta la clave 'allowed_file_formats' en el config.")

            if not isinstance(dataset_conf['partition_columns'], list) or not dataset_conf['partition_columns']:
                raise TypeError(
                    f"El campo 'partition_columns' en el dataset '{dataset}' de la fuente '{data_source}' debe ser una lista no vacía."
                )
        except Exception as e:
            logger.exception(f"Error en LandingBatchReader.validate_config: {e}")
            raise

    @classmethod
    def from_config(
        cls,
        config_path: str,
        data_source: str,
        dataset: str
    ) -> "LandingBatchReader":
        """
        Crea una instancia a partir de un archivo de configuración JSON.
        Args:
            config_path: Ruta al archivo de configuración.
            data_source: Nombre de la fuente de datos.
            dataset: Nombre del dataset.
        Returns:
            Instancia de LandingBatchReader.
        """
        try:
            logger.info(f"Leyendo configuración desde: {config_path}")
            with open(config_path, 'r') as f:
                config = json.load(f)
            logger.info(f"Configuración leída correctamente para {data_source}/{dataset}")
            LandingBatchReader.validate_config(config, data_source, dataset)
            return cls(
                data_source=data_source,
                dataset=dataset,
                landing_path=config['data_type_information'][data_source][dataset]['landing_path'],
                raw_path=config['data_type_information'][data_source][dataset]['raw_path'],
                bronze_path=config['data_type_information'][data_source][dataset]['bronze_path'],
                dataset_format=config['data_type_information'][data_source][dataset]['dataset_format'],
                schema_json=config['data_type_information'][data_source][dataset].get('schema_json'),
                allowed_file_formats=config['allowed_file_formats'],
                processing_type=config['data_type_information'][data_source][dataset]['processing_type'],
            )
        except Exception as e:
            logger.exception(f"Error en LandingBatchReader.from_config: {e}")
            raise

    @staticmethod
    def add_image_metadata(df: DataFrame) -> DataFrame:
        def extract_label(path_col):
            return F.regexp_extract(path_col, "flower_photos/([^/]+)", 1)

        def extract_size(content):
            image = Image.open(io.BytesIO(content))
            return image.size

        @F.pandas_udf("width: int, height: int")
        def extract_size_udf(content_series):
            sizes = content_series.apply(extract_size)
            return pd.DataFrame(list(sizes))

        return (
            df.withColumn("_size", extract_size_udf(F.col("content")))
            .withColumn("_width", F.col("_size.width"))
            .withColumn("_height", F.col("_size.height"))
            .withColumn("_label", extract_label(F.col("path")))
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_ingested_file", F.input_file_name())
            .drop("_size")
        )

    def add_metadata_columns(self, df: DataFrame) -> DataFrame:
        """
        Añade columnas de metadatos y de particionado temporal al DataFrame.
        Args:
            df: DataFrame de entrada.
        Returns:
            DataFrame con columnas adicionales de metadatos.
        Raises:
            ValueError: Si no existe la columna 'timestamp' en el DataFrame.
        """
        try:
            if self.format == "binary":
                df = self.add_image_metadata(df)
            data_cols = df.columns
            if "timestamp" not in data_cols:
                logger.error("Falta columna 'timestamp' en el DataFrame.")
                raise ValueError(
                    "Error: el DataFrame no contiene la columna timestamp. "
                    "Revisa el esquema de entrada.")

            metadata_cols = ['_ingested_at', '_ingested_filename', 'year', 'month', 'day', 'hour']

            logger.info("Añadiendo columnas de metadatos al DataFrame.")
            df = (df.withColumn("_ingested_at", F.current_timestamp())
                  .withColumn("_ingested_filename", F.input_file_name())
                  .withColumn("year", F.year("timestamp"))
                  .withColumn("month", F.month("timestamp"))
                  .withColumn("day", F.dayofmonth("timestamp"))
                  .withColumn("hour", F.hour("timestamp"))
                  )
            return df.select(metadata_cols + data_cols)
        except Exception as e:
            logger.exception(f"Error en LandingBatchReader.add_metadata_columns: {e}")
            logger.exception(f"Error en LandingBatchReader.add_metadata_columns: {e}")
            raise


    def from_batch_read(self) -> DataFrame:
        """
        Lee datos en streaming desde la zona landing usando Auto Loader (cloudFiles).
        Returns:
            DataFrame en streaming listo para procesamiento.
        """
        try:
            logger.info(f"Iniciando lectura desde Auto Loader en formato {self.format}")
            dbutils.fs.mkdirs(self.bronze_schema_location)
            if self.format == "csv":
                return (spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format", self.format)
                        .option("header", "true")
                        .schema(self.schema_json)
                        .option("recursiveFileLookup", "true")
                        .option("cloudFiles.schemaLocation", self.bronze_schema_location)
                        .option("rescuedDataColumn", "_rescued_data")
                        .load(self.landing_path)
                        )
            elif self.format == "binary":
                return (
                    spark.readStream
                    .format("cloudFiles")
                    .option("cloudFiles.format", "binaryFile")
                    .option("pathGlobFilter", "*.jpg")
                    .option("recursiveFileLookup", "true")
                    .option("cloudFiles.schemaLocation", self.bronze_schema_location)
                    .load(self.landing_path)
                )
            elif self.format in {"json", "avro", "parquet"}:
                return (spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format", self.format)
                        .schema(self.schema_json)
                        .option("cloudFiles.schemaLocation", self.bronze_schema_location)
                        .option("recursiveFileLookup", "true")
                        .load(self.landing_path)
                        )
            else:
                raise ValueError(f"Formato de archivo '{self.format}' no soportado.")
        except Exception as e:
            logger.exception(f"Error en LandingBatchReader.read_autoloader: {e}")
            raise

    def read(self) -> DataFrame:
        """
        Ejecuta la lectura en streaming y añade metadatos.
        Returns:
            DataFrame de streaming con columnas de metadatos añadidas.
        Raises:
            Exception: Si el formato no es soportado.
        """
        try:
            logger.info(f"Ejecutando lectura para dataset '{self.dataset}' con formato '{self.format}'")
            df = self.from_batch_read()
            logger.info("Añadiendo columnas de metadatos al DataFrame leído.")
            return df.transform(self.add_metadata_columns)
        except Exception as e:
            logger.exception(f"Error en LandingBatchReader.read: {e}")
            raise

class BronzeBatchWriter:
    """
    Clase encargada de escribir un DataFrame batch en la capa bronze,
    aplicando particionamiento, checkpoint, persistencia en Delta Lake
    y registro opcional en el catálogo Hive Metastore
    """

    def __init__(
            self,
            data_source: str,
            dataset: str,
            landing_path: str,
            raw_path: str,
            bronze_path: str,
            partition_columns: list,
            processing_type: str
    ):
        """
        Inicializa el escritor de la capa bronze.
        Args:
            data_source (str): Nombre de la fuente de datos.
            dataset (str): Nombre del dataset.
            landing_path (str): Ruta de almacenamiento landing.
            raw_path (str): Ruta de almacenamiento raw.
            bronze_path (str): Ruta de almacenamiento bronze.
            partition_columns (list): Columnas para particionar el almacenamiento.
            processing_type (str): Tipo de procesamiento, puede ser 'batch' o 'streaming'.
        """
        self.data_source = data_source
        self.dataset = dataset
        self.landing_path = landing_path
        self.raw_path = raw_path
        self.bronze_path = bronze_path
        self.checkpoint_location = f"{bronze_path}_checkpoint"
        self.table = f"hive_metastore.bronze.{self.data_source}_{self.dataset}"
        self.query_name = f"bronze-{data_source}-{dataset}"
        self.partition_columns = partition_columns
        self.processing_type = processing_type
        logger.info(f"BronzeWriter inicializado para {data_source}/{dataset} en modo {processing_type}")

    def __str__(self) -> str:
        """
        Devuelve una representación legible de la instancia.
        """
        return (
            f"[BronzeWriter] dataset='{self.dataset}' source='{self.data_source}' "
            f"table='{self.table}' checkpoint='{self.checkpoint_location}' "
            f"partitions={self.partition_columns} "
            f"paths: landing='{self.landing_path}', raw='{self.raw_path}', bronze='{self.bronze_path}' "
            f"mode={self.processing_type}"
        )

    @classmethod
    def from_config(cls, config_path: str, data_source: str, dataset: str) -> "BronzeWriter":
        """
        Crea una instancia de BronzeWriter a partir de un archivo JSON de configuración.
        Args:
            config_path (str): Ruta al archivo de configuración.
            data_source (str): Nombre de la fuente de datos.
            dataset (str): Nombre del dataset.
        Returns:
            BronzeWriter: Instancia configurada.
        """
        try:
            logger.info(f"Leyendo configuración desde: {config_path}")
            with open(config_path, 'r') as f:
                config = json.load(f)
            logger.info(f"Configuración cargada para {data_source}/{dataset}")
            return cls(
                data_source=data_source,
                dataset=dataset,
                landing_path=config['data_type_information'][data_source][dataset]['landing_path'],
                raw_path=config['data_type_information'][data_source][dataset]['raw_path'],
                bronze_path=config['data_type_information'][data_source][dataset]['bronze_path'],
                partition_columns=config['data_type_information'][data_source][dataset]['partition_columns'],
                processing_type=config['data_type_information'][data_source][dataset]['processing_type']
            )
        except Exception as e:
            logger.exception(f"Error en BronzeWriter.from_config: {e}")
            raise

    def archive_raw_files(self, df: DataFrame) -> None:
        """
        Mueve los archivos desde landing hacia la zona raw si existe la columna '_ingested_filename'.
        Args:
            df (DataFrame): DataFrame del batch o microbatch a procesar.
        """
        try:
            logger.info("Archivando archivos desde landing hacia raw...")
            if "_ingested_filename" in df.columns:
                files = [row["_ingested_filename"] for row in df.select("_ingested_filename").distinct().collect()]
                for file in files:
                    if file:
                        destino = file.replace(self.landing_path, self.raw_path)
                        logger.info(f"Moviendo archivo: {file} → {destino}")
                        dbutils.fs.mkdirs(destino.rsplit("/", 1)[0])
                        dbutils.fs.mv(file, destino)
                logger.info("Archivos archivados correctamente.")
            else:
                logger.warning("No se encontró la columna '_ingested_filename'. No se moverá ningún archivo.")
        except Exception as e:
            logger.exception(f"Error en BronzeWriter.archive_raw_files: {e}")
            raise

    def write_to_bronze(self, df: DataFrame) -> None:
        """
        Realiza la escritura del DataFrame en formato Delta en la capa bronze,
        particionando según columnas configuradas. Soporta batch o streaming según configuración.
        Args:
            df (DataFrame): DataFrame a escribir.
        """
        try:
            logger.info("Escribiendo a bronze...")
            dbutils.fs.mkdirs(self.bronze_path)
            spark.sql('CREATE DATABASE IF NOT EXISTS hive_metastore.bronze')
            (df.write
                 .format("delta")
                 .option("mergeSchema", "true")
                 .option("path", self.bronze_path)
                 .partitionBy(*self.partition_columns)
                 .mode("append")
                 .save()
             )

            spark.sql(f"CREATE TABLE IF NOT EXISTS {self.table} USING DELTA LOCATION '{self.bronze_path}'")
            logger.info(f"Escritura completada y tabla registrada: {self.table}")
        except Exception as e:
            logger.exception(f"Error en BronzeWriter.write_to_bronze: {e}")
            raise

    def append_2_bronze(self, df: DataFrame, df_id: Optional[int] = None) -> None:
        """
        Aplica el flujo completo de escritura: escritura en bronze, archivado y limpieza.
        Args:
            df (DataFrame): DataFrame a procesar.
            df_id (Optional[int]): ID del batch o microbatch, útil para logs.
        """
        try:
            if df_id is not None:
                logger.info(f"Procesando batch {df_id}")
            df.persist()

            logger.debug("Ejecutando escritura en bronze")
            self.write_to_bronze(df)

            logger.debug("Archivando archivos crudos")
            self.archive_raw_files(df)

            df.unpersist()
            logger.info(f"Batch {df_id} procesado correctamente")
        except Exception as e:
            logger.exception(f"Error en BronzeWriter.append: {e}")
            raise

INFO:py4j.clientserver:Received command c on object id p1


**INGESTA DE TIPO BATCH**

AQUI PROBAMOS LA INGESTA TIPO BATCH PARA:   
  

     => retail/sales_order  

     => inventory/inventario  

     => logistics/shipments  

     => weather/forecast-data

In [0]:
config_path = "/Volumes/masterdbr_tema8/default/config/azure_config.json"

INFO:py4j.clientserver:Received command c on object id p1


In [0]:
data_source = "retail"
dataset = "sales_orders"

reader = LandingBatchReader.from_config(config_path, data_source, dataset)
writer = BronzeBatchWriter.from_config(config_path, data_source, dataset)

query = (reader
  .read()
  .writeStream
  .foreachBatch(writer.append_2_bronze)
  .trigger(availableNow=True)
  .option("checkpointLocation", writer.checkpoint_location)
  .queryName(writer.query_name)
  .start()
)

print(query.status)
print(query.lastProgress)
print(query.exception())

INFO:py4j.clientserver:Received command c on object id p1
INFO:__main__:Leyendo configuración desde: /Volumes/masterdbr_tema8/default/config/azure_config.json
INFO:__main__:Configuración leída correctamente para retail/sales_orders
INFO:__main__:Leyendo configuración desde: /Volumes/masterdbr_tema8/default/config/azure_config.json
INFO:__main__:Configuración cargada para retail/sales_orders
INFO:__main__:BronzeWriter inicializado para retail/sales_orders en modo batch
INFO:__main__:Ejecutando lectura para dataset 'sales_orders' con formato 'json'
INFO:__main__:Iniciando lectura desde Auto Loader en formato json
INFO:py4j.clientserver:Received command c on object id p0
INFO:__main__:Añadiendo columnas de metadatos al DataFrame leído.
INFO:__main__:Añadiendo columnas de metadatos al DataFrame.
INFO:py4j.clientserver:Received command c on object id p0


{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
None
None


In [0]:
data_source = "inventory"
dataset = "inventario"

reader = LandingBatchReader.from_config(config_path, data_source, dataset)
writer = BronzeBatchWriter.from_config(config_path, data_source, dataset)

query = (reader
  .read()
  .writeStream
  .foreachBatch(writer.append_2_bronze)
  .trigger(availableNow=True)
  .option("checkpointLocation", writer.checkpoint_location)
  .queryName(writer.query_name)
  .start()
)

print(query.status)
print(query.lastProgress)
print(query.exception())

INFO:py4j.clientserver:Received command c on object id p1
INFO:py4j.clientserver:Received command c on object id p0
INFO:__main__:Leyendo configuración desde: /Volumes/masterdbr_tema8/default/config/azure_config.json
INFO:__main__:Configuración leída correctamente para inventory/inventario
INFO:__main__:Leyendo configuración desde: /Volumes/masterdbr_tema8/default/config/azure_config.json
INFO:__main__:Configuración cargada para inventory/inventario
INFO:__main__:BronzeWriter inicializado para inventory/inventario en modo batch
INFO:__main__:Ejecutando lectura para dataset 'inventario' con formato 'csv'
INFO:__main__:Iniciando lectura desde Auto Loader en formato csv
INFO:py4j.clientserver:Received command c on object id p0
INFO:__main__:Añadiendo columnas de metadatos al DataFrame leído.
INFO:__main__:Añadiendo columnas de metadatos al DataFrame.


{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
None
None


In [0]:
data_source = "logistics"
dataset = "shipments"

reader = LandingBatchReader.from_config(config_path, data_source, dataset)
writer = BronzeBatchWriter.from_config(config_path, data_source, dataset)

query = (reader
  .read()
  .writeStream
  .foreachBatch(writer.append_2_bronze)
  .trigger(availableNow=True)
  #.trigger(processingTime="60 seconds") # modo continuo
  .option("checkpointLocation", writer.checkpoint_location)
  .queryName(writer.query_name)
  .start()
)

print(query.status)
print(query.lastProgress)
print(query.exception())

INFO:py4j.clientserver:Received command c on object id p1
INFO:__main__:Leyendo configuración desde: /Volumes/masterdbr_tema8/default/config/azure_config.json
INFO:py4j.clientserver:Received command c on object id p0
INFO:__main__:Configuración leída correctamente para logistics/shipments
INFO:__main__:Leyendo configuración desde: /Volumes/masterdbr_tema8/default/config/azure_config.json
INFO:__main__:Configuración cargada para logistics/shipments
INFO:__main__:BronzeWriter inicializado para logistics/shipments en modo batch
INFO:__main__:Ejecutando lectura para dataset 'shipments' con formato 'csv'
INFO:__main__:Iniciando lectura desde Auto Loader en formato csv
INFO:__main__:Añadiendo columnas de metadatos al DataFrame leído.
INFO:__main__:Añadiendo columnas de metadatos al DataFrame.
INFO:py4j.clientserver:Python Server ready to receive messages
INFO:py4j.clientserver:Received command c on object id p13
INFO:py4j.clientserver:Received command c on object id p0
INFO:__main__:Procesan

{'message': 'Initializing StreamExecution', 'isDataAvailable': False, 'isTriggerActive': False}
None
None


In [0]:
data_source = "weather"
dataset = "forecast_data"

reader = LandingBatchReader.from_config(config_path, data_source, dataset)
writer = BronzeBatchWriter.from_config(config_path, data_source, dataset)

query = (reader
  .read()
  .writeStream
  .foreachBatch(writer.append_2_bronze)
  .trigger(availableNow=True)
  #.trigger(processingTime="60 seconds") # modo continuo
  .option("checkpointLocation", writer.checkpoint_location)
  .queryName(writer.query_name)
  .start()
)

print(query.status)
print(query.lastProgress)
print(query.exception())

INFO:py4j.clientserver:Received command c on object id p1
INFO:__main__:Leyendo configuración desde: /Volumes/masterdbr_tema8/default/config/azure_config.json
INFO:py4j.clientserver:Received command c on object id p0
INFO:__main__:Configuración leída correctamente para weather/forecast_data
INFO:__main__:Leyendo configuración desde: /Volumes/masterdbr_tema8/default/config/azure_config.json
INFO:__main__:Configuración cargada para weather/forecast_data
INFO:__main__:BronzeWriter inicializado para weather/forecast_data en modo batch
INFO:__main__:Ejecutando lectura para dataset 'forecast_data' con formato 'json'
INFO:__main__:Iniciando lectura desde Auto Loader en formato json
INFO:__main__:Añadiendo columnas de metadatos al DataFrame leído.
INFO:__main__:Añadiendo columnas de metadatos al DataFrame.
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Python Server ready to receive messages
INFO:py4j.clientserver:Received command c on object id p14
INFO:__mai

{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
None
None


INFO:__main__:Escribiendo a bronze...
